# Custom Decoder-Only Model

# 1️⃣ Start Coding a Minimal Version

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleDecoderBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, ff_size):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden_size, num_heads)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, ff_size),
            nn.ReLU(),
            nn.Linear(ff_size, hidden_size)
        )
        self.ln2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        # x shape: seq_len, batch, hidden
        attn_out, _ = self.attn(x, x, x, attn_mask=self._causal_mask(x.size(0)))
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ff(x))
        return x

    def _causal_mask(self, size):
        # mask out future positions
        mask = torch.triu(torch.ones(size, size) * float('-inf'), 1)
        return mask

class SimpleDecoderModel(nn.Module):
    def __init__(self, vocab_size, hidden_size=64, num_heads=2, ff_size=128, num_layers=2, max_len=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden_size)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.layers = nn.ModuleList([
            SimpleDecoderBlock(hidden_size, num_heads, ff_size) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        seq_len, batch = x.size()
        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(seq_len).unsqueeze(1))
        h = tok + pos
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        logits = self.head(h)
        return logits


# 2️⃣ Train on Toy Data

In [13]:
data = [
    "hello world",
    "hi there",
    "hey you",
    "good morning",
    "good night",
    "how are you",
    "what is up",
    "have fun",
    "see you",
    "take care",
    "thank you",
    "sorry about that",
    "nice to meet you",
    "welcome back",
    "long time no see",
    "how is it going",
    "what are you doing",
    "i am fine",
    "all the best",
    "have a nice day",
    "let us go",
    "come here",
    "go away",
    "stay safe",
    "be happy",
    "call me",
    "text me",
    "see you soon",
    "good luck",
    "cheers"
]

In [14]:
# Example: learning sequences like "hello"
vocab = list("abcdefghijklmnopqrstuvwxyz ")
token2id = {c:i for i,c in enumerate(vocab)}
id2token = {i:c for c,i in token2id.items()}

def encode(s):
    return [token2id[c] for c in s]

# data = ["hello world", "hi there", "hey you"]
inputs = [torch.tensor(encode(d)).unsqueeze(1) for d in data]  # seq_len x batch
targets = [torch.tensor(encode(d)).unsqueeze(1) for d in data]

# Training loop
model = SimpleDecoderModel(vocab_size=len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    total_loss = 0
    for x, y in zip(inputs, targets):
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.view(-1, len(vocab)), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}: {total_loss:.4f}")


Epoch 0: 72.1729
Epoch 1: 31.5756
Epoch 2: 13.7841
Epoch 3: 6.6766
Epoch 4: 3.7568
Epoch 5: 2.4384
Epoch 6: 1.7533
Epoch 7: 1.3471
Epoch 8: 1.0596
Epoch 9: 0.8668
Epoch 10: 0.7259
Epoch 11: 0.6189
Epoch 12: 0.5352
Epoch 13: 0.4683
Epoch 14: 0.4137
Epoch 15: 0.3685
Epoch 16: 0.3306
Epoch 17: 0.2984
Epoch 18: 0.2709
Epoch 19: 0.2471
Epoch 20: 0.2264
Epoch 21: 0.2083
Epoch 22: 0.1923
Epoch 23: 0.1781
Epoch 24: 0.1654
Epoch 25: 0.1541
Epoch 26: 0.1439
Epoch 27: 0.1347
Epoch 28: 0.1263
Epoch 29: 0.1187
Epoch 30: 0.1118
Epoch 31: 0.1054
Epoch 32: 0.0996
Epoch 33: 0.0942
Epoch 34: 0.0893
Epoch 35: 0.0847
Epoch 36: 0.0805
Epoch 37: 0.0765
Epoch 38: 0.0729
Epoch 39: 0.0695
Epoch 40: 0.0663
Epoch 41: 0.0633
Epoch 42: 0.0605
Epoch 43: 0.0579
Epoch 44: 0.0555
Epoch 45: 0.0532
Epoch 46: 0.0510
Epoch 47: 0.0490
Epoch 48: 0.0470
Epoch 49: 0.0452
Epoch 50: 0.0435
Epoch 51: 0.0418
Epoch 52: 0.0403
Epoch 53: 0.0388
Epoch 54: 0.0374
Epoch 55: 0.0361
Epoch 56: 0.0348
Epoch 57: 0.0336
Epoch 58: 0.0325
Epoc

# 3️⃣ Prepare the Model for Generation

In [15]:
import torch
import torch.nn.functional as F

def generate(model, start_seq, token2id, id2token, max_len=20):
    model.eval()  # set to evaluation mode
    seq = [token2id[c] for c in start_seq]
    for _ in range(max_len - len(seq)):
        x = torch.tensor(seq).unsqueeze(1)  # seq_len x batch=1
        with torch.no_grad():
            logits = model(x)
        next_token_logits = logits[-1, 0]  # last token logits
        next_token_id = torch.argmax(next_token_logits).item()  # greedy
        seq.append(next_token_id)
        if id2token[next_token_id] == " ":
            break  # optional: stop on space
    return "".join(id2token[i] for i in seq)

# 4️⃣ Feed Toy Sequences

In [17]:
vocab = list("abcdefghijklmnopqrstuvwxyz ")
token2id = {c:i for i,c in enumerate(vocab)}
id2token = {i:c for i,c in enumerate(vocab)}

start = "he"
generated = generate(model, start, token2id, id2token, max_len=10)
print("Generated:", generated)

Generated: heeeeeeeee


# Interactive Playground

In [18]:
while True:
    start = input("Type start sequence: ")
    if not start or start == 'x':
        break
    print("Generated:", generate(model, start, token2id, id2token, max_len=20))

Type start sequence: h
Generated: hhhhhhhhhhhhhhhhhhhh
Type start sequence: hi
Generated: hiiiiiiiiiiiiiiiiiii
Type start sequence: no
Generated: nooooooooooooooooooo
Type start sequence: aj
Generated: aj 
Type start sequence: nai
Generated: naiiiiiiiiiiiiiiiiii
Type start sequence: is
Generated: isssssssssssssssssss
Type start sequence: x


# Visualize Attention (Optional Fun Step)

In [19]:
x_raw = torch.tensor(encode("hello")).unsqueeze(1) # Original input: (seq_len, batch)
seq_len = x_raw.size(0)

# Replicate the embedding steps from SimpleDecoderModel's forward method
tok_emb = model.token_emb(x_raw)
pos_emb = model.pos_emb(torch.arange(seq_len, device=x_raw.device).unsqueeze(1))

# This `h` is the correctly embedded input for the attention layer
h = tok_emb + pos_emb # Shape: (seq_len, batch, hidden_size)

attn_layer = model.layers[0].attn
attn_output, attn_weights = attn_layer(h, h, h, attn_mask=model.layers[0]._causal_mask(h.size(0)))
print(attn_weights.shape)

torch.Size([1, 5, 5])
